# MCP Server Patterns — Live, Cell-by-Cell Demo
### Workshop project #3 | Teal Trust

This notebook is the **runnable, notebook version of the "MCP Server Patterns" project** (the
`app.py` Gradio app). Instead of a web UI, you step through each pattern as a cell and watch the
**JSON-RPC 2.0 wire protocol** move between client and server.

**The problem MCP solves — N×M → N+M.** Before MCP, every agent needed bespoke code to talk to
every tool. MCP makes a tool expose itself **once** as a server, and any MCP-capable agent can use
it. Same client code, any server.

```
   +-------------+   JSON-RPC 2.0     +--------------------+
   |   Claude    |   over stdio       |   MCP SERVER(s)    |
   | (the LLM)   | <----------------> |   tools            |
   | ReAct loop  |   initialize       |   resources        |
   | (agent.py)  |   tools/list       |   prompts          |
   +-------------+   tools/call       +--------------------+
                     resources/read
                     prompts/get
```

MCP has **three primitives**: **tools** (imperative actions, like POST), **resources** (read-only
data, like GET), and **prompts** (reusable templates). Two advanced features round it out:
**sampling** (the server borrows the client's LLM) and **stateless vs stateful** servers.

### What we'll build and run

| Step | Cell does |
|---|---|
| **Setup** | imports, optional API key, the server framework, the client, the ReAct agent |
| **Servers** | write four real MCP servers to `mcp_servers/` (general, math, banking, stateful) |
| **Patterns 1–8** | run each pattern; every cell prints the MCP wire trace |
| **Compare / Status** | same question across patterns; live server processes |

> **API key:** the *MCP mechanics* (list/call tools, read resources, get prompts, stateless vs
> stateful) run with **no key**. The *agentic* patterns use real Claude when
> `ANTHROPIC_API_KEY` is set, and fall back to direct tool calls when it isn't — so every cell
> runs in class either way.

## Setup — imports, folders, and the (optional) API key

Standard library only, plus two optional packages:
```
pip install python-dotenv anthropic
```
We create an `mcp_servers/` folder (the servers will be written there) and detect a key without
quitting if it's missing.

In [ ]:
import os, sys, json, time, subprocess, threading, sqlite3, textwrap

# Optional .env loading.
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# Optional Anthropic SDK (only the agentic patterns need it).
try:
    import anthropic
except Exception:
    anthropic = None

API_KEY = os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("CLAUDE_API_KEY", "")
MODEL   = os.environ.get("CLAUDE_MODEL", "claude-sonnet-4-6")   # also: claude-opus-4-8, claude-haiku-4-5

ROOT = os.getcwd()
SERVERS = os.path.join(ROOT, "mcp_servers")
os.makedirs(SERVERS, exist_ok=True)

def have_key():
    return bool(API_KEY) and anthropic is not None

print("Working dir :", ROOT)
print("Servers dir :", SERVERS)
if have_key():
    print(f"API key found -> agentic patterns will call real Claude ({MODEL}).")
else:
    print("No API key (or anthropic not installed) -> agentic patterns fall back to direct tool calls.")
    print("The MCP mechanics still run fully.")

## The MCP wire contract (recap)

Every MCP message is **JSON-RPC 2.0**. A request is `{"jsonrpc":"2.0","id":N,"method":...,"params":...}`;
a response is `{"jsonrpc":"2.0","id":N,"result":...}` (or `"error"`). The client only ever needs
these seven methods:

| Method | Meaning |
|---|---|
| `initialize` | handshake — who are you, which protocol version |
| `tools/list` · `tools/call` | discover tools · run one |
| `resources/list` · `resources/read` | discover data · read one |
| `prompts/list` · `prompts/get` | discover templates · render one |

We build the servers **from scratch** (no SDK) so all of this is visible on the wire. In
production you'd swap in the official `mcp` / `FastMCP` SDK — the message shapes are identical.

## Part 1 · The server framework — `_mcp_base.py`

A tiny base class every server reuses. You register tools/resources/prompts with decorators
(`@srv.tool(...)`, `@srv.resource(...)`, `@srv.prompt(...)`), and one `handle()` method answers all
seven JSON-RPC methods. `serve_forever()` is the stdio loop: read one JSON line, reply one JSON
line. Read `handle()` once and the whole protocol is demystified.

In [ ]:
# Write _mcp_base.py to mcp_servers/, then show its source.
server_src = r"""
# _mcp_base.py  --  minimal MCP server framework (JSON-RPC 2.0 over stdio).
# Zero dependencies. Register tools/resources/prompts, then serve_forever().
# The message shapes follow the MCP spec, so a real MCP client can talk to it.
import sys, json

class MCPServer:
    def __init__(self, name, version="1.0.0"):
        self.name = name
        self.version = version
        self._tools = {}       # name -> {description, inputSchema, fn}
        self._resources = {}   # uri  -> {description, fn}
        self._prompts = {}     # name -> {description, arguments, fn}

    # ---- registration decorators ----
    def tool(self, name, description, input_schema):
        def deco(fn):
            self._tools[name] = {"description": description, "inputSchema": input_schema, "fn": fn}
            return fn
        return deco

    def resource(self, uri, description):
        def deco(fn):
            self._resources[uri] = {"description": description, "fn": fn}
            return fn
        return deco

    def prompt(self, name, description, arguments):
        def deco(fn):
            self._prompts[name] = {"description": description, "arguments": arguments, "fn": fn}
            return fn
        return deco

    # ---- JSON-RPC helpers ----
    def _ok(self, rid, result):
        return {"jsonrpc": "2.0", "id": rid, "result": result}

    def _err(self, rid, code, message):
        return {"jsonrpc": "2.0", "id": rid, "error": {"code": code, "message": message}}

    # ---- the single request handler (the whole protocol lives here) ----
    def handle(self, req):
        method = req.get("method")
        rid = req.get("id")
        params = req.get("params") or {}

        if method == "initialize":
            return self._ok(rid, {
                "protocolVersion": "2025-06-18",
                "serverInfo": {"name": self.name, "version": self.version},
                "capabilities": {"tools": {}, "resources": {}, "prompts": {}}})

        if method == "tools/list":
            return self._ok(rid, {"tools": [
                {"name": n, "description": d["description"], "inputSchema": d["inputSchema"]}
                for n, d in self._tools.items()]})

        if method == "tools/call":
            name = params.get("name"); args = params.get("arguments") or {}
            if name not in self._tools:
                return self._err(rid, -32601, "tool not found: " + str(name))
            try:
                text = self._tools[name]["fn"](**args)
                return self._ok(rid, {"content": [{"type": "text", "text": str(text)}], "isError": False})
            except Exception as e:
                return self._ok(rid, {"content": [{"type": "text", "text": "error: " + str(e)}], "isError": True})

        if method == "resources/list":
            return self._ok(rid, {"resources": [
                {"uri": u, "description": d["description"]} for u, d in self._resources.items()]})

        if method == "resources/read":
            uri = params.get("uri")
            if uri not in self._resources:
                return self._err(rid, -32601, "resource not found: " + str(uri))
            text = self._resources[uri]["fn"]()
            return self._ok(rid, {"contents": [{"uri": uri, "mimeType": "text/plain", "text": str(text)}]})

        if method == "prompts/list":
            return self._ok(rid, {"prompts": [
                {"name": n, "description": d["description"],
                 "arguments": [{"name": a, "required": True} for a in d["arguments"]]}
                for n, d in self._prompts.items()]})

        if method == "prompts/get":
            name = params.get("name"); args = params.get("arguments") or {}
            if name not in self._prompts:
                return self._err(rid, -32601, "prompt not found: " + str(name))
            text = self._prompts[name]["fn"](**args)
            return self._ok(rid, {"messages": [
                {"role": "user", "content": {"type": "text", "text": str(text)}}]})

        return self._err(rid, -32601, "unknown method: " + str(method))

    def serve_forever(self):
        # Read one JSON request per line from stdin; write one JSON response per line to stdout.
        for line in sys.stdin:
            line = line.strip()
            if not line:
                continue
            try:
                req = json.loads(line)
            except Exception:
                continue
            sys.stdout.write(json.dumps(self.handle(req)) + "\n")
            sys.stdout.flush()
"""

with open(os.path.join(SERVERS, "_mcp_base.py"), "w") as f:
    f.write(server_src)
print("Wrote mcp_servers/_mcp_base.py  (" + str(server_src.count(chr(10))) + " lines)")
print(server_src)

## Part 2 · The MCP client — `MCPClient`

The client spawns a server as a **subprocess** and speaks JSON-RPC 2.0 to it over stdin/stdout. It
records **every message** into a `trace` list, which is exactly what lets us print the wire
protocol after each demo. (In production you'd use the SDK's `ClientSession`; the messages are the
same.)

In [ ]:
class MCPClient:
    def __init__(self, server_path, env_extra=None, name=None):
        self.server_path = server_path
        self.name = name or os.path.basename(server_path)
        self.env_extra = env_extra or {}
        self.proc = None
        self._id = 0
        self._lock = threading.Lock()
        self.trace = []            # list of {"dir": "->"/"<-", "msg": {...}}
        self.started_at = None

    # ---- lifecycle ----
    def start(self):
        env = dict(os.environ); env.update(self.env_extra)
        self.proc = subprocess.Popen(
            [sys.executable, self.server_path],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            stderr=subprocess.PIPE, text=True, bufsize=1, env=env)
        self.started_at = time.time()
        self._request("initialize", {
            "protocolVersion": "2025-06-18", "capabilities": {},
            "clientInfo": {"name": "demo-client", "version": "1.0"}})
        return self

    def stop(self):
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try: self.proc.wait(timeout=3)
            except Exception: self.proc.kill()

    def is_alive(self):
        return self.proc is not None and self.proc.poll() is None

    @property
    def pid(self):
        return self.proc.pid if self.proc else None

    @property
    def uptime(self):
        return round(time.time() - self.started_at, 1) if self.started_at else 0

    # ---- one JSON-RPC round-trip ----
    def _request(self, method, params=None, timeout=15):
        with self._lock:
            self._id += 1
            req = {"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}}
            self.trace.append({"dir": "->", "msg": req})
            try:
                self.proc.stdin.write(json.dumps(req) + "\n")
                self.proc.stdin.flush()
                line = self.proc.stdout.readline()
                if not line:
                    err = {"error": "no response (server may have crashed)"}
                    self.trace.append({"dir": "<-", "msg": err}); return err
                resp = json.loads(line.strip())
                self.trace.append({"dir": "<-", "msg": resp}); return resp
            except Exception as e:
                err = {"error": f"transport error: {e}"}
                self.trace.append({"dir": "<-", "msg": err}); return err

    # ---- MCP primitives (thin wrappers over _request) ----
    def list_tools(self):
        return self._request("tools/list").get("result", {}).get("tools", [])

    def call_tool(self, name, arguments):
        r = self._request("tools/call", {"name": name, "arguments": arguments})
        if "error" in r:
            return json.dumps(r["error"])
        return "\n".join(c.get("text", "") for c in r.get("result", {}).get("content", []))

    def list_resources(self):
        return self._request("resources/list").get("result", {}).get("resources", [])

    def read_resource(self, uri):
        r = self._request("resources/read", {"uri": uri})
        return "\n".join(c.get("text", "") for c in r.get("result", {}).get("contents", []))

    def list_prompts(self):
        return self._request("prompts/list").get("result", {}).get("prompts", [])

    def get_prompt(self, name, arguments):
        r = self._request("prompts/get", {"name": name, "arguments": arguments})
        return "\n".join(m.get("content", {}).get("text", "")
                         for m in r.get("result", {}).get("messages", []))

    def clear_trace(self):
        self.trace = []


print("MCPClient defined.")

### Small helpers to view the results inline

`print_trace` renders the JSON-RPC wire messages, `print_log` renders the agent's ReAct steps
(THINK / ACT / OBSERVE), and `print_answer` shows the final answer plus token usage.

In [ ]:
def print_trace(client, last=20):
    print("\n----- MCP wire trace (JSON-RPC 2.0) -----")
    if not client or not client.trace:
        print("(no messages yet)"); return
    for e in client.trace[-last:]:
        arrow = "CLIENT --> SERVER" if e["dir"] == "->" else "SERVER --> CLIENT"
        print(arrow)
        print(json.dumps(e["msg"], indent=2))
        print()

def print_log(log):
    icon = {"thought": "THINK", "action": "ACT", "observation": "OBSERVE", "info": "INFO"}
    print("\n----- ReAct steps -----")
    for kind, content in log:
        c = content
        if kind == "observation" and len(str(c)) > 400:
            c = str(c)[:400] + " ...(truncated)"
        print(f"[{icon.get(kind, kind.upper())}] {c}")

def print_answer(r):
    print("ANSWER:\n" + str(r.get("answer", "")))
    u = r.get("usage") or {}
    if u:
        print(f"\n[tokens in/out: {u.get('input_tokens',0)}/{u.get('output_tokens',0)}"
              f"  est. cost ~${r.get('cost_usd',0):.5f}]")

print("Print helpers ready.")

## Part 3 · Write the four MCP servers

Each server is a small standalone program that imports `_mcp_base` and registers its primitives.
The next four cells write them into `mcp_servers/`. Read each one — they're short, and they're the
whole point of "how each type of MCP server works."

### 3a · `general_server.py` — all three primitives (Patterns 1, 4, 5)

Two **tools** (`convert_units`, `get_stock_quote`), two **resources**
(`policy://kyc-summary`, `customer://C-1001`), and one **prompt** template (`structured_analysis`).

In [ ]:
# Write general_server.py to mcp_servers/, then show its source.
server_src = r"""
# general_server.py  --  Patterns 1/4/5: ONE server exposing all THREE primitives.
from _mcp_base import MCPServer

srv = MCPServer("general-server")

# ---- TOOLS (imperative actions, like POST) ----
LEN = {"m": 1.0, "km": 1000.0, "mile": 1609.344, "miles": 1609.344, "ft": 0.3048, "feet": 0.3048}

@srv.tool("convert_units", "Convert a length between units (m, km, mile, ft).",
          {"type": "object",
           "properties": {"value": {"type": "number"},
                          "from_unit": {"type": "string"},
                          "to_unit": {"type": "string"}},
           "required": ["value", "from_unit", "to_unit"]})
def convert_units(value, from_unit, to_unit):
    fu, tu = str(from_unit).lower(), str(to_unit).lower()
    if fu not in LEN or tu not in LEN:
        return "unsupported unit(s): " + str(from_unit) + ", " + str(to_unit)
    meters = float(value) * LEN[fu]
    result = meters / LEN[tu]
    return str(value) + " " + from_unit + " = " + format(result, ".4f") + " " + to_unit

@srv.tool("get_stock_quote", "Return a (mock) delayed stock quote for a ticker symbol.",
          {"type": "object", "properties": {"symbol": {"type": "string"}}, "required": ["symbol"]})
def get_stock_quote(symbol):
    price = 500 + (abs(hash(str(symbol).upper())) % 2000)
    return str(symbol).upper() + ": INR " + format(price, ",") + ".00 (mock delayed quote)"

# ---- RESOURCES (read-only data, like GET) ----
@srv.resource("policy://kyc-summary", "One-paragraph KYC policy summary.")
def kyc_summary():
    return ("KYC policy: government photo ID + address proof are mandatory. "
            "PAN required for transactions above INR 50,000. Re-verify every 24 months.")

@srv.resource("customer://C-1001", "Static profile card for customer C-1001.")
def customer_c1001():
    return "C-1001 | Arjun Mehta | premium/current | Mumbai | KYC verified"

# ---- PROMPT (reusable template with variable slots) ----
@srv.prompt("structured_analysis", "Template: structured analysis of a topic for an audience.",
            ["topic", "audience"])
def structured_analysis(topic, audience="a general audience"):
    return ("Give a structured analysis of " + str(topic) + " for " + str(audience) + ". "
            "Use three sections: (1) what it is, (2) why it matters, (3) key trade-offs. "
            "Keep each section to 3 bullet points.")

srv.serve_forever()
"""

with open(os.path.join(SERVERS, "general_server.py"), "w") as f:
    f.write(server_src)
print("Wrote mcp_servers/general_server.py  (" + str(server_src.count(chr(10))) + " lines)")
print(server_src)

### 3b · `math_server.py` — a second server for fan-out (Pattern 3)

Just tools: `compound_interest` and `simple_interest`. Its only job is to exist as a *second*
server so Claude has to route each subtask to the server that owns the right tool.

In [ ]:
# Write math_server.py to mcp_servers/, then show its source.
server_src = r"""
# math_server.py  --  Pattern 3: a SECOND server so Claude can fan out across servers.
from _mcp_base import MCPServer

srv = MCPServer("math-server")

@srv.tool("compound_interest", "Compound interest: final amount and interest for P at rate% for years.",
          {"type": "object",
           "properties": {"principal": {"type": "number"},
                          "rate": {"type": "number"},
                          "years": {"type": "number"},
                          "compounds_per_year": {"type": "integer"}},
           "required": ["principal", "rate", "years"]})
def compound_interest(principal, rate, years, compounds_per_year=1):
    P = float(principal); r = float(rate) / 100.0; n = int(compounds_per_year); t = float(years)
    amount = P * (1 + r / n) ** (n * t)
    interest = amount - P
    return ("Compound interest on " + format(P, ",.0f") + " at " + str(rate) + "% for "
            + str(years) + "y (n=" + str(n) + "): amount = " + format(amount, ",.2f")
            + ", interest = " + format(interest, ",.2f"))

@srv.tool("simple_interest", "Simple interest for P at rate% for years.",
          {"type": "object",
           "properties": {"principal": {"type": "number"},
                          "rate": {"type": "number"},
                          "years": {"type": "number"}},
           "required": ["principal", "rate", "years"]})
def simple_interest(principal, rate, years):
    P = float(principal); interest = P * float(rate) / 100.0 * float(years)
    return "Simple interest = " + format(interest, ",.2f") + " (on principal " + format(P, ",.0f") + ")"

srv.serve_forever()
"""

with open(os.path.join(SERVERS, "math_server.py"), "w") as f:
    f.write(server_src)
print("Wrote mcp_servers/math_server.py  (" + str(server_src.count(chr(10))) + " lines)")
print(server_src)

### 3c · Seed the banking database, then `banking_server.py` (Pattern 8)

The banking server is backed by a real SQLite database. First we seed `banking.db` (three accounts,
five transactions — one deliberately flagged), then write the server that reads it.

In [ ]:
# Seed banking.db (same data as the project's db_seed.py) into the notebook root.
BANKING_DB = os.path.join(ROOT, "banking.db")
if os.path.exists(BANKING_DB):
    os.remove(BANKING_DB)
conn = sqlite3.connect(BANKING_DB); c = conn.cursor()
c.executescript("""
CREATE TABLE accounts (
    customer_id TEXT PRIMARY KEY, name TEXT, segment TEXT,
    account_type TEXT, balance REAL, city TEXT, kyc_status TEXT);
CREATE TABLE transactions (
    txn_id INTEGER PRIMARY KEY, customer_id TEXT, txn_date TEXT,
    type TEXT, amount REAL, description TEXT);
""")
c.executemany("INSERT INTO accounts VALUES (?,?,?,?,?,?,?)", [
    ("C-1001", "Arjun Mehta",  "premium",   "current", 245000,  "Mumbai",  "verified"),
    ("C-1002", "Priya Nair",   "retail",    "savings",  18500,  "Chennai", "verified"),
    ("C-1003", "Deepak Shah",  "corporate", "current", 1200000, "Delhi",   "pending"),
])
c.executemany("INSERT INTO transactions VALUES (?,?,?,?,?,?)", [
    (1, "C-1001", "2026-06-10", "credit", 50000, "Salary credit"),
    (2, "C-1001", "2026-06-08", "debit",  12000, "Utility payment"),
    (3, "C-1001", "2026-06-05", "debit",  85000, "Wire transfer - flagged"),
    (4, "C-1002", "2026-06-09", "credit",  8000, "Freelance payment"),
    (5, "C-1003", "2026-06-07", "credit", 300000, "Invoice settlement"),
])
conn.commit(); conn.close()
print("Seeded", BANKING_DB)

In [ ]:
# Write banking_server.py to mcp_servers/, then show its source.
server_src = r"""
# banking_server.py  --  Pattern 8: a DOMAIN server backed by SQLite (banking.db).
import os, sqlite3
from _mcp_base import MCPServer

# Find the database: an env override first, else banking.db one level up from this file.
DB = os.environ.get("BANKING_DB") or os.path.join(
    os.path.dirname(os.path.abspath(__file__)), "..", "banking.db")

def q(sql, args=()):
    conn = sqlite3.connect(DB); conn.row_factory = sqlite3.Row
    rows = [dict(r) for r in conn.execute(sql, args).fetchall()]
    conn.close(); return rows

srv = MCPServer("banking-server")

@srv.tool("get_account", "Look up a customer's account by customer_id.",
          {"type": "object", "properties": {"customer_id": {"type": "string"}},
           "required": ["customer_id"]})
def get_account(customer_id):
    rows = q("SELECT * FROM accounts WHERE customer_id=?", (customer_id,))
    if not rows:
        return "No account for " + str(customer_id)
    a = rows[0]
    return (a["customer_id"] + " | " + a["name"] + " | " + a["segment"] + "/" + a["account_type"]
            + " | balance INR " + format(a["balance"], ",.0f") + " | " + a["city"]
            + " | KYC " + a["kyc_status"])

@srv.tool("get_transactions", "Recent transactions for a customer (newest first).",
          {"type": "object",
           "properties": {"customer_id": {"type": "string"}, "limit": {"type": "integer"}},
           "required": ["customer_id"]})
def get_transactions(customer_id, limit=5):
    rows = q("SELECT * FROM transactions WHERE customer_id=? ORDER BY txn_date DESC LIMIT ?",
             (customer_id, int(limit)))
    if not rows:
        return "No transactions for " + str(customer_id)
    return "\n".join(r["txn_date"] + "  " + r["type"] + "  INR " + format(r["amount"], ",.0f")
                     + "  " + r["description"] for r in rows)

@srv.tool("fraud_risk_score", "Heuristic fraud-risk score (0-99) for a customer.",
          {"type": "object", "properties": {"customer_id": {"type": "string"}},
           "required": ["customer_id"]})
def fraud_risk_score(customer_id):
    rows = q("SELECT * FROM transactions WHERE customer_id=?", (customer_id,))
    if not rows:
        return "No transactions for " + str(customer_id)
    score = 10; signals = []
    for t in rows:
        if t["amount"] >= 80000:
            score += 35; signals.append("large txn " + format(t["amount"], ",.0f"))
        if "flag" in (t["description"] or "").lower():
            score += 40; signals.append("flagged: " + t["description"])
    score = min(score, 99)
    band = "HIGH" if score >= 70 else ("MEDIUM" if score >= 40 else "LOW")
    return ("Fraud risk for " + str(customer_id) + ": " + str(score) + "/100 (" + band + "). "
            + "Signals: " + (", ".join(signals) if signals else "none"))

@srv.resource("policy://basel-iii", "Basel III capital-adequacy summary.")
def basel_iii():
    return ("Basel III: minimum CET1 4.5%, Tier 1 6%, total capital 8% of risk-weighted assets; "
            "capital conservation buffer 2.5%; LCR >= 100%; NSFR >= 100%.")

@srv.prompt("compliance_review", "Template: AML/KYC compliance review note for a customer.",
            ["customer_id"])
def compliance_review(customer_id):
    return ("Draft an AML/KYC compliance review for customer " + str(customer_id) + ". "
            "Check KYC status, flag transactions above regulatory thresholds, and recommend "
            "actions consistent with Basel III and RBI norms.")

srv.serve_forever()
"""

with open(os.path.join(SERVERS, "banking_server.py"), "w") as f:
    f.write(server_src)
print("Wrote mcp_servers/banking_server.py  (" + str(server_src.count(chr(10))) + " lines)")
print(server_src)

### 3d · `stateful_server.py` — stateless vs stateful toggle (Pattern 7)

One server, two behaviours chosen by the `MCP_MODE` environment variable. In **stateful** mode it
remembers the current customer between calls; in **stateless** mode it deliberately forgets — the
production-recommended design, because stateless servers scale like ordinary web services.

In [ ]:
# Write stateful_server.py to mcp_servers/, then show its source.
server_src = r"""
# stateful_server.py  --  Pattern 7: toggle stateless vs stateful via the MCP_MODE env var.
import os
from _mcp_base import MCPServer

MODE = os.environ.get("MCP_MODE", "stateless")
srv = MCPServer("stateful-server[" + MODE + "]")

_session = {"current_customer": None}   # only consulted in stateful mode

@srv.tool("set_current_customer", "Set the current customer for this session.",
          {"type": "object", "properties": {"customer_id": {"type": "string"}},
           "required": ["customer_id"]})
def set_current_customer(customer_id):
    if MODE == "stateful":
        _session["current_customer"] = customer_id
        return "[stateful] current customer stored server-side: " + str(customer_id)
    return "[stateless] received " + str(customer_id) + " but stateless servers keep no session state."

@srv.tool("get_current_customer", "Return the current session customer.",
          {"type": "object", "properties": {}})
def get_current_customer():
    if MODE == "stateful":
        return "[stateful] current customer is " + str(_session["current_customer"])
    return "[stateless] no session memory - the client must pass customer_id on every call."

srv.serve_forever()
"""

with open(os.path.join(SERVERS, "stateful_server.py"), "w") as f:
    f.write(server_src)
print("Wrote mcp_servers/stateful_server.py  (" + str(server_src.count(chr(10))) + " lines)")
print(server_src)

## Part 4 · The ReAct agent — `agent.py`

The agent is the loop that lets **Claude** drive the servers: discover tools, send them to Claude,
and whenever Claude emits a `tool_use`, route it to the server that owns that tool, run it, and
feed the result back — repeating until Claude is done. It also includes the **remote MCP connector**
(Pattern 2) and the **sampling** helper (Pattern 6).

*These functions only run when an API key is set; the pattern cells below guard on `have_key()`.*

In [ ]:
_llm = None
def llm():
    """Lazily create the Anthropic client (only when an agentic pattern actually runs)."""
    global _llm
    if _llm is None:
        if anthropic is None:
            raise RuntimeError("anthropic package not installed")
        _llm = anthropic.Anthropic(api_key=API_KEY)
    return _llm

MAX_ITERS = 8

def mcp_tools_to_anthropic(mcp_clients):
    """Build Anthropic tools[] and a name->client routing map from one or more MCP servers."""
    tools, route = [], {}
    for mc in mcp_clients:
        for t in mc.list_tools():
            tools.append({"name": t["name"], "description": t["description"],
                          "input_schema": t["inputSchema"]})
            route[t["name"]] = mc
    return tools, route

def run_agent(question, model, mcp_clients, system=None):
    """ReAct loop across one or more MCP servers. Returns dict with answer + step log."""
    tools, route = mcp_tools_to_anthropic(mcp_clients)
    messages = [{"role": "user", "content": question}]
    log, answer = [], ""
    usage = {"input_tokens": 0, "output_tokens": 0}
    sys_prompt = system or ("You are a helpful assistant. Use the available MCP tools when they "
                            "give better or real-time data. Explain briefly which tool you use and why.")

    for _ in range(MAX_ITERS):
        resp = llm().messages.create(
            model=model, max_tokens=1500, system=sys_prompt,
            tools=tools if tools else [], messages=messages)
        usage["input_tokens"]  += resp.usage.input_tokens
        usage["output_tokens"] += resp.usage.output_tokens

        for b in resp.content:
            if b.type == "text" and b.text.strip():
                log.append(("thought", b.text.strip())); answer = b.text.strip()

        if resp.stop_reason != "tool_use":
            break

        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type != "tool_use":
                continue
            mc = route.get(b.name)
            srv_name = mc.name if mc else "?"
            log.append(("action", f"call {b.name} on [{srv_name}]  args={json.dumps(b.input)}"))
            out = json.dumps({"error": f"no server owns tool {b.name}"}) if mc is None \
                  else mc.call_tool(b.name, b.input)
            log.append(("observation", out))
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": out})
        messages.append({"role": "user", "content": results})

    cost = usage["input_tokens"]/1e6*3 + usage["output_tokens"]/1e6*15   # sonnet pricing
    return {"answer": answer, "log": log, "usage": usage, "cost_usd": round(cost, 6)}

print("run_agent ready.")

In [ ]:
def run_remote_mcp(question, model, server_url, token=None):
    """Pattern 2: Anthropic beta MCP connector — Claude connects to a REMOTE server server-side."""
    if not server_url:
        return {"answer": "No REMOTE_MCP_URL configured. Set it in .env to try a live remote MCP server.",
                "log": [("info", "Remote MCP demo skipped -- no URL set.")], "usage": {}, "cost_usd": 0}
    mcp_server = {"type": "url", "url": server_url, "name": "remote-mcp"}
    if token:
        mcp_server["authorization_token"] = token
    try:
        resp = llm().beta.messages.create(
            model=model, max_tokens=1200,
            messages=[{"role": "user", "content": question}],
            mcp_servers=[mcp_server], betas=["mcp-client-2025-11-20"])
        answer = " ".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
        return {"answer": answer or "(no text)",
                "log": [("info", f"Connected to remote MCP: {server_url}"),
                        ("observation", "Tool discovery + calls handled server-side by Anthropic.")],
                "usage": {"input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens},
                "cost_usd": 0}
    except Exception as e:
        return {"answer": f"Remote MCP error: {e}",
                "log": [("info", f"Failed to reach {server_url}: {e}")], "usage": {}, "cost_usd": 0}

def mcp_sampling_demo(text_to_summarise, model):
    """Pattern 6: the SERVER asks the CLIENT to run an LLM completion (sampling/createMessage)."""
    sampling_request = {
        "method": "sampling/createMessage",
        "params": {"messages": [{"role": "user",
                                  "content": {"type": "text",
                                              "text": f"Summarise in one sentence: {text_to_summarise}"}}],
                   "maxTokens": 100, "modelPreferences": {"hints": [{"name": model}]}}}
    log = [("action", "SERVER -> CLIENT  sampling/createMessage"),
           ("observation", json.dumps(sampling_request, indent=2))]
    try:
        resp = llm().messages.create(
            model=model, max_tokens=120,
            messages=[{"role": "user",
                       "content": sampling_request["params"]["messages"][0]["content"]["text"]}])
        summary = " ".join(b.text for b in resp.content if b.type == "text")
        log.append(("thought", "CLIENT fulfils the sampling request with a real Claude call."))
        log.append(("observation", f"CLIENT -> SERVER  result: {summary}"))
        return {"answer": summary, "log": log,
                "usage": {"input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens},
                "cost_usd": 0}
    except Exception as e:
        return {"answer": f"Sampling error: {e}", "log": log, "usage": {}, "cost_usd": 0}

print("run_remote_mcp and mcp_sampling_demo ready.")

### Long-lived clients

Like the real app, we start each server once and reuse it. `get_client(key, filename, env_extra)`
spawns the server on first use and caches it. Run the cleanup cell at the very end to stop them.

In [ ]:
CLIENTS = {}
def get_client(key, filename, env_extra=None):
    c = CLIENTS.get(key)
    if c is None or not c.is_alive():
        c = MCPClient(os.path.join(SERVERS, filename), env_extra=env_extra, name=key).start()
        CLIENTS[key] = c
    return c

print("get_client ready. Servers will start on first use.")

---
# The eight MCP server patterns

Each pattern below is a markdown explainer followed by one runnable cell. Every cell clears the
trace first, so the wire log you see is exactly that pattern's traffic.

| # | Pattern | Primitives | Server(s) |
|---|---|---|---|
| 1 | Local stdio server | tools + resources + prompts | general |
| 2 | Remote MCP connector (beta) | server-side discovery + call | remote URL |
| 3 | Multi-server orchestration | tools across 2 servers | general + math |
| 4 | Resources (data pull) | resources/list, resources/read | general |
| 5 | Prompt templates | prompts/list, prompts/get | general |
| 6 | Sampling | sampling/createMessage | (simulated) |
| 7 | Stateless vs stateful | tools + session contrast | stateful |
| 8 | Banking domain server | tools + resource + prompt + SQLite | banking |

## Pattern 1 · Local stdio server *(all three primitives)*

A single custom server launched as a local subprocess. Claude discovers its tools, then answers a
question that needs two of them. The wire trace shows `initialize` → `tools/list` → `tools/call`.

In [ ]:
question = "Convert 100 km to miles and give me a stock quote for INFY."
c = get_client("general", "general_server.py"); c.clear_trace()
print("Tools on general-server:", [t["name"] for t in c.list_tools()])

if have_key():
    r = run_agent(question, MODEL, [c]); print_answer(r); print_log(r["log"])
else:
    print("\n(No API key -> calling the tools directly so you still see them run)")
    print("  ", c.call_tool("convert_units", {"value": 100, "from_unit": "km", "to_unit": "miles"}))
    print("  ", c.call_tool("get_stock_quote", {"symbol": "INFY"}))

print_trace(c)

## Pattern 2 · Remote MCP connector *(Anthropic beta)*

Claude connects to a **remote** MCP server server-side — it discovers and calls tools without your
code proxying them. This needs `REMOTE_MCP_URL` set in `.env`; without it the cell shows the
graceful "skipped" path so you can see the shape of the call.

In [ ]:
url = os.environ.get("REMOTE_MCP_URL", ""); tok = os.environ.get("REMOTE_MCP_TOKEN", "")
if have_key() or not url:
    r = run_remote_mcp("What tools does this remote MCP server expose?", MODEL, url, tok)
    print_answer(r); print_log(r["log"])
else:
    print("A remote URL is set but no API key -> the connector call needs a key. Skipping the live call.")

## Pattern 3 · Multi-server orchestration *(fan-out)*

Two servers at once. Claude sees a merged tool list and routes each subtask to the server that owns
the right tool: `convert_units` on **general**, `compound_interest` on **math**. Watch the `ACT`
lines name the server for each call.

In [ ]:
question = "Convert 5 km to miles, then compute compound interest on 100000 at 8% for 5 years."
gen = get_client("general", "general_server.py"); gen.clear_trace()
mth = get_client("math", "math_server.py"); mth.clear_trace()
print("general tools:", [t["name"] for t in gen.list_tools()])
print("math tools   :", [t["name"] for t in mth.list_tools()])

if have_key():
    r = run_agent(question, MODEL, [gen, mth]); print_answer(r); print_log(r["log"])
else:
    print("\n(No API key -> calling one tool on EACH server directly)")
    print("  ", gen.call_tool("convert_units", {"value": 5, "from_unit": "km", "to_unit": "miles"}))
    print("  ", mth.call_tool("compound_interest", {"principal": 100000, "rate": 8, "years": 5}))

print("\n===== general-server trace ====="); print_trace(gen)
print("\n===== math-server trace ====="); print_trace(mth)

## Pattern 4 · Resources *(read-only data — no LLM needed)*

Resources are a **declarative data pull**, like an HTTP `GET`: no action, no side effect. The client
lists them and reads them directly. This whole pattern runs without an API key.

In [ ]:
c = get_client("general", "general_server.py"); c.clear_trace()
print("Resources on general-server:", [r["uri"] for r in c.list_resources()])
for uri in ["policy://kyc-summary", "customer://C-1001"]:
    print("\n" + uri + " ->")
    print("  ", c.read_resource(uri))
print_trace(c)

## Pattern 5 · Prompt templates *(reusable, centrally managed)*

A prompt is a server-provided template with variable slots. The client renders it with arguments
via `prompts/get`. The render needs no key; if a key is set we also send the rendered prompt to
Claude.

In [ ]:
c = get_client("general", "general_server.py"); c.clear_trace()
print("Prompts on general-server:", [p["name"] for p in c.list_prompts()])
rendered = c.get_prompt("structured_analysis",
                        {"topic": "the benefits of MCP servers", "audience": "beginners"})
print("\nRendered template:\n" + rendered)

if have_key():
    r = run_agent(rendered, MODEL, [])          # no tools — just run the prompt
    print("\nClaude's response:\n" + r["answer"])

print_trace(c)

## Pattern 6 · Sampling *(server borrows the client's LLM — reverse direction)*

Normally the client calls the server. **Sampling flips it:** while running a tool, the server sends
`sampling/createMessage` back to the *client*, which fulfils it with its own LLM. With a key we run
the real round-trip; without one we print the exact request the server would send.

In [ ]:
text = ("MCP is an open standard that lets any AI agent connect to any tool or data source "
        "through one universal protocol.")
if have_key():
    r = mcp_sampling_demo(text, MODEL); print_answer(r); print_log(r["log"])
else:
    print("(No API key -> showing the sampling request the SERVER would send to the CLIENT)")
    sampling_request = {
        "method": "sampling/createMessage",
        "params": {"messages": [{"role": "user",
                                  "content": {"type": "text", "text": f"Summarise in one sentence: {text}"}}],
                   "maxTokens": 100, "modelPreferences": {"hints": [{"name": MODEL}]}}}
    print(json.dumps(sampling_request, indent=2))
    print("\nThe client would run its LLM on that request and return the summary to the server.")

## Pattern 7 · Stateless vs stateful *(architecture contrast)*

The **same** server, launched twice with different `MCP_MODE`. We call `set_current_customer` then
`get_current_customer` in each mode. Stateful *remembers*; stateless *forgets* by design — which is
why stateless servers scale horizontally like ordinary web services. No API key needed.

In [ ]:
for mode in ["stateless", "stateful"]:
    c = get_client("stateful_" + mode, "stateful_server.py", env_extra={"MCP_MODE": mode})
    c.clear_trace()
    print("===== MODE:", mode, "=====")
    print("  set:", c.call_tool("set_current_customer", {"customer_id": "C-1001"}))
    print("  get:", c.call_tool("get_current_customer", {}))
    print()

# Show one trace so you can see the JSON-RPC for the stateful case.
print_trace(get_client("stateful_stateful", "stateful_server.py", env_extra={"MCP_MODE": "stateful"}))

## Pattern 8 · Banking domain server *(tools + resource + prompt + SQLite)*

A realistic domain server: account lookup, transactions, and a fraud-risk score, all reading the
seeded SQLite DB — plus a Basel III resource and a compliance-review prompt. With a key, Claude
chains the tools to answer; without one we call them directly.

In [ ]:
c = get_client("banking", "banking_server.py",
               env_extra={"BANKING_DB": os.path.join(ROOT, "banking.db")}); c.clear_trace()
print("Banking tools:", [t["name"] for t in c.list_tools()])
question = "Look up customer C-1001, show their recent transactions, and give me their fraud risk score."

if have_key():
    r = run_agent(question, MODEL, [c],
                  system="You are a banking assistant. Use the banking MCP tools to answer. "
                         "Be precise with figures and cite the customer_id.")
    print_answer(r); print_log(r["log"])
else:
    print("\n(No API key -> calling the banking tools directly)")
    print(c.call_tool("get_account", {"customer_id": "C-1001"}))
    print(c.call_tool("get_transactions", {"customer_id": "C-1001"}))
    print(c.call_tool("fraud_risk_score", {"customer_id": "C-1001"}))
    print("\nResource policy://basel-iii ->")
    print("  ", c.read_resource("policy://basel-iii"))

print_trace(c)

## Compare patterns 1 / 3 / 8 on one question

The same banking question, answered three ways. With a key this runs the agent on each and
tabulates tokens/cost; without one it shows the direct banking answer (the pure-MCP result).

In [ ]:
question = "What is customer C-1001's fraud risk score?"
bank = get_client("banking", "banking_server.py",
                  env_extra={"BANKING_DB": os.path.join(ROOT, "banking.db")}); bank.clear_trace()

if have_key():
    gen = get_client("general", "general_server.py"); gen.clear_trace()
    r1 = run_agent(question, MODEL, [gen])
    gen.clear_trace(); mth = get_client("math", "math_server.py"); mth.clear_trace()
    r3 = run_agent(question, MODEL, [gen, mth])
    r8 = run_agent(question, MODEL, [bank])
    print("| Pattern | Answer (short) | In tok | Out tok | Cost |")
    print("|---|---|---|---|---|")
    for name, r in [("P1 single", r1), ("P3 multi", r3), ("P8 banking", r8)]:
        print(f"| {name} | {r['answer'][:50]}... | {r['usage'].get('input_tokens',0)} "
              f"| {r['usage'].get('output_tokens',0)} | ${r['cost_usd']:.5f} |")
else:
    print("(No API key -> the pure-MCP answer from the banking server:)")
    print("  ", bank.call_tool("fraud_risk_score", {"customer_id": "C-1001"}))
    print("\nWith a key, this cell runs the same question through patterns 1, 3, and 8 and compares cost.")

## Server status *(live processes)*

Like the app's Status tab: which server subprocesses are alive, their PID, uptime, and how many
tools / resources / prompts each exposes.

In [ ]:
def server_status():
    if not CLIENTS:
        print("No servers started yet -- run a pattern cell first."); return
    print(f"{'server':22} {'alive':6} {'pid':7} {'uptime':8} tools/res/prompts")
    print("-" * 60)
    for key, c in CLIENTS.items():
        if c.is_alive():
            nt = len(c.list_tools()); nr = len(c.list_resources()); npr = len(c.list_prompts())
            print(f"{key:22} {'yes':6} {str(c.pid):7} {str(c.uptime):8} {nt}/{nr}/{npr}")
        else:
            print(f"{key:22} {'no':6} {'-':7} {'-':8} -")

server_status()

## Cleanup — stop the server subprocesses

Run this when you're done to terminate every spawned server cleanly.

In [ ]:
for key, c in list(CLIENTS.items()):
    try: c.stop()
    except Exception: pass
print("Stopped", len(CLIENTS), "MCP server subprocess(es).")

---
## Recap — the one page to remember

**MCP = one JSON-RPC 2.0 contract** (`initialize`, `tools/*`, `resources/*`, `prompts/*`) that any
client and any server agree on. Write a capability once as a server; every MCP-capable agent can use
it — that's the N×M → N+M collapse.

**Three primitives, by control model**

| Primitive | Like | Who controls | Example here |
|---|---|---|---|
| **Tools** | POST | the model decides to call | `convert_units`, `fraud_risk_score` |
| **Resources** | GET | the app pulls data in | `policy://kyc-summary`, `policy://basel-iii` |
| **Prompts** | a saved template | the user picks it | `structured_analysis`, `compliance_review` |

**Two advanced moves**

- **Sampling** — the server borrows the *client's* LLM (reverse direction). One server, any model.
- **Stateless vs stateful** — stateless servers keep no session memory, so they scale like web
  services. Prefer stateless; pass what you need on every call.

**The pattern behind every cell:** build a JSON-RPC request → the client sends it over stdio → the
server's `handle()` answers → read the result. Transports and primitives change; that loop doesn't.

**From here to production:** swap the hand-built servers for the official `mcp` / `FastMCP` SDK
(`@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`) — the wire messages are identical, so your
client and agent code don't change.

**Homework:** add a `get_exchange_rate(base, quote)` tool to `general_server.py`, re-run Pattern 1,
and watch it appear in `tools/list` with no client or agent changes — tools are discovered at
runtime.